In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results


In [12]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Generator\n\nHere's a simple and clean solution:\n\n```python\ndef get_s3_uri(bucket_name: str, object_key: str) -> str:\n    \"\"\"\n    Generate an S3 URI from a bucket name and object key.\n    \n    Args:\n        bucket_name (str): The name of the S3 bucket\n        object_key (str): The key/path of the object in the bucket\n        \n    Returns:\n        str: The S3 URI in the format 's3://bucket-name/object-key'\n        \n    Raises:\n        ValueError: If bucket_name or object_key is empty\n    \"\"\"\n    if not bucket_name or not isinstance(bucket_name, str):\n        raise ValueError(\"bucket_name must be a non-empty string\")\n    if not object_key or not isinstance(object_key, str):\n        raise ValueError(\"object_key must be a non-empty string\")\n    \n    return f\"s3://{bucket_name}/{object_key}\"\n\n\n# Example usage\nif __name__ == \"__main__\":\n    # Basic example\n    uri = get_s3_uri(\"my-bucket\", \"path/to/file.txt\")\n  